# ML-07 — Baseline Action Score + Frozen Multi-Task Benchmarks

Assignment 5 baseline for the locked content-performance prioritisation POC. The literal card requirement is the transparent ranking rule; the project methodology also requires frozen classification and regression baselines so Assignment 6 can compare all learned components against pre-existing honest benchmarks.


## 1. Signal checks + rule reasoning

### Signal 1 — content age / staleness

**Hypothesis:** older content may be a stronger candidate for refresh. This is the required flag-linked signal check because staleness sits behind FlyRank refresh logic.

The feature itself was already selected in Assignment 4 without using April. Here, April is used only as the future outcome for evaluating that pre-selected March-safe signal.

Age buckets are fixed in advance for interpretability: **0–90**, **91–180**, **181–365**, and **366+ days**. The executed table prints `n` for every bucket.

**Verdict: MIXED**

The broad direction supports a staleness effect, but not cleanly enough to call it confirmed. The youngest pages have the lowest observed decline rate (**55.72%**) and the oldest pages the highest (**84.96%**), while the two middle buckets reverse order (**72.08%** for 91–180 days versus **65.81%** for 181–365 days). Median future impression change is also most negative for the 366+ day bucket. Staleness is therefore informative directionally, but a simple monotonic 'older always means worse' assumption is not supported by these buckets.

### Signal 2 — CTR relative to search position

**Hypothesis:** pages with unusually weak CTR for their March search position may be stronger candidates for CTR/content-fix review.

This follows the FlyRank CTR-fix idea while staying inside the Assignment 4 contract. Both `aggregate_ctr` and `median_position` are March-only features already selected before April was examined.

Pages are first placed into fixed March position bands: **1–3**, **4–10**, **11–20**, and **21+**. The first attempted normalization divided each page's CTR by its position-band median, but the executed March data showed at least one band has a median CTR of zero, making that ratio invalid. The audit therefore uses a safer March-only relative measure: each page's **CTR percentile rank among pages in the same position band**. Those ranks are bucketed using fixed quartiles: **bottom 25%**, **25–50%**, **50–75%**, and **top 25%**. April is still used only to evaluate the pre-defined signal.

**Verdict: CONFIRMED**

The result is directionally ordered across all four March-only CTR-relative-to-position buckets. Observed future decline falls from **74.66%** in the bottom quartile to **71.14%**, **65.46%**, and **52.33%** in the top quartile. Median future impression change also improves monotonically from **−34.70%** to **−30.12%**, **−25.30%**, and **−4.71%**. Mean future change moves in the same direction and becomes positive in the upper half. This supports the hypothesis that unusually weak CTR relative to comparable ranking position is a useful review signal.

The March diagnostic also explains why percentile ranking is preferable to a CTR/median ratio here: the **21+ position band has 63.75% zero-CTR pages and a median CTR of zero**, so a ratio to the median would be undefined. The within-band percentile construction handles that observed data shape without using April to tune the signal.

With both signal verdicts now locked — **staleness: MIXED** and **CTR relative to position: CONFIRMED** — the next step is to encode one transparent baseline rule.

### One-rule baseline

The two signal audits imply a deliberately simple rule:

> **Queue a page for review when its March CTR is in the bottom 25% of pages with a similar March search position. Give the page 2 points for meeting that low-CTR condition, and add 1 extra point if the page is 366+ days old. Rank higher scores first, then weaker within-position CTR first.**

Why this shape:

- **CTR relative to position is the gate** because its signal verdict was **CONFIRMED**.
- **Staleness is only a priority boost**, not a gate, because its signal verdict was **MIXED**. The oldest bucket still showed the clearest weakness, so 366+ days can break priority without pretending that age is monotonic across all buckets.
- The point values are fixed, human-readable rule points — **not fitted weights**.
- The queue uses March-only inputs. April is used only afterward to evaluate how often the rule's picks declined.

The rule emits exactly one reason code: `LOW_CTR_FOR_POSITION`.

The action label is: `REVIEW_CTR_REFRESH`.


In [1]:
# STEP 1A — reconstruct the locked Assignment-4 POC and audit staleness.
# This cell intentionally uses the exact same population rules as w03_data_contract.ipynb.

import os
import duckdb
import numpy as np
import pandas as pd

# Hugging Face token stays private: read from environment / Colab Secrets only.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is missing. Add a Hugging Face READ token as a Colab Secret named HF_TOKEN."
    )

con = duckdb.connect()
safe_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# 1) Exact longitudinal eligibility from Assignment 4: >=20 usable GSC days in both months.
march_cov = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           COUNT(DISTINCT report_date) AS march_usable_days
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

april_cov = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           COUNT(DISTINCT report_date) AS april_usable_days
    FROM {APRIL}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

matched_keys = (
    march_cov[march_cov["march_usable_days"] >= 20]
    .merge(april_cov, on=["client_hash_id", "content_hash_id"], how="inner")
)
matched_keys = matched_keys[matched_keys["april_usable_days"] >= 20][
    ["client_hash_id", "content_hash_id"]
].drop_duplicates()
con.register("matched_keys", matched_keys)

# 2) Reapply the locked March exposure strata used only for balanced sampling.
march_exposure = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        SUM(f.gsc_impressions)::DOUBLE / COUNT(DISTINCT f.report_date)
            AS march_avg_impressions_per_day
    FROM {MARCH} AS f
    INNER JOIN matched_keys AS k USING (client_hash_id, content_hash_id)
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()

march_exposure["exposure_tier"] = pd.cut(
    march_exposure["march_avg_impressions_per_day"],
    bins=[-float("inf"), 13.42, 58.25, float("inf")],
    labels=["Low", "Medium", "High"],
    include_lowest=True,
)

client_tier_counts = (
    march_exposure.groupby(["client_hash_id", "exposure_tier"], observed=False)
    .size().unstack(fill_value=0)
    .reindex(columns=["Low", "Medium", "High"], fill_value=0)
)
eligible_clients = client_tier_counts[
    client_tier_counts.min(axis=1) >= 40
].index.tolist()

balanced_poc = (
    march_exposure[march_exposure["client_hash_id"].isin(eligible_clients)]
    .sort_values(["client_hash_id", "exposure_tier", "content_hash_id"])
    .groupby(["client_hash_id", "exposure_tier"], observed=False, group_keys=False)
    .head(40)
    .reset_index(drop=True)
)

# 3) March-safe staleness feature: page age as known on 31 March 2026.
balanced_keys = balanced_poc[["client_hash_id", "content_hash_id"]].drop_duplicates()
con.register("balanced_keys", balanced_keys)

age_frame = con.sql(f"""
    SELECT
        d.client_hash_id,
        d.content_hash_id,
        DATE_DIFF('day', d.content_created_date, DATE '2026-03-31')::DOUBLE
            AS content_age_days
    FROM {DIM_CONTENT} AS d
    INNER JOIN balanced_keys AS k USING (client_hash_id, content_hash_id)
""").df()

# 4) Future outcome only: March -> April relative change in average impressions per usable day.
march_target = con.sql(f"""
    SELECT f.client_hash_id, f.content_hash_id,
           SUM(f.gsc_impressions)::DOUBLE / COUNT(DISTINCT f.report_date)
               AS march_avg_impressions_per_day
    FROM {MARCH} AS f
    INNER JOIN balanced_keys AS k USING (client_hash_id, content_hash_id)
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()

april_target = con.sql(f"""
    SELECT f.client_hash_id, f.content_hash_id,
           SUM(f.gsc_impressions)::DOUBLE / COUNT(DISTINCT f.report_date)
               AS april_avg_impressions_per_day
    FROM {APRIL} AS f
    INNER JOIN balanced_keys AS k USING (client_hash_id, content_hash_id)
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()

signal_frame = (
    age_frame
    .merge(march_target, on=["client_hash_id", "content_hash_id"], how="inner")
    .merge(april_target, on=["client_hash_id", "content_hash_id"], how="inner")
)
signal_frame["future_impression_change"] = (
    signal_frame["april_avg_impressions_per_day"]
    - signal_frame["march_avg_impressions_per_day"]
) / signal_frame["march_avg_impressions_per_day"]
signal_frame["future_decline"] = signal_frame["future_impression_change"] < 0

signal_frame["age_bucket"] = pd.cut(
    signal_frame["content_age_days"],
    bins=[-float("inf"), 90, 180, 365, float("inf")],
    labels=["0-90 days", "91-180 days", "181-365 days", "366+ days"],
)

staleness_table = (
    signal_frame.groupby("age_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        decline_rate=("future_decline", "mean"),
        median_future_impression_change=("future_impression_change", "median"),
        mean_future_impression_change=("future_impression_change", "mean"),
    )
    .reset_index()
)
staleness_table["decline_rate_pct"] = 100 * staleness_table.pop("decline_rate")

print("Locked POC clients:", balanced_poc["client_hash_id"].nunique())
print("Locked POC pages:", len(balanced_poc))
print("Expected: 21 clients and 2,520 pages")
print("Signal rows:", len(signal_frame))
print("\nSTALENESS SIGNAL TABLE")
display(staleness_table)


# STEP 1B — audit CTR relative to March search position.
# Uses March-only aggregate CTR and March-only median position.
# April remains outcome/evaluation only.

march_ctr_position = con.sql(f"""
    WITH daily AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            f.gsc_clicks::DOUBLE AS clicks,
            f.gsc_impressions::DOUBLE AS impressions,
            CASE
                WHEN f.gsc_avg_position >= 1
                THEN f.gsc_avg_position::DOUBLE
                ELSE NULL
            END AS valid_position
        FROM {MARCH} AS f
        INNER JOIN balanced_keys AS k USING (client_hash_id, content_hash_id)
        WHERE f.gsc_data_available IS TRUE
    )
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(clicks) / NULLIF(SUM(impressions), 0) AS aggregate_ctr,
        MEDIAN(valid_position) AS median_position
    FROM daily
    GROUP BY client_hash_id, content_hash_id
""").df()

ctr_signal = signal_frame[
    ["client_hash_id", "content_hash_id", "future_impression_change", "future_decline"]
].merge(
    march_ctr_position,
    on=["client_hash_id", "content_hash_id"],
    how="inner",
)

ctr_signal["position_band"] = pd.cut(
    ctr_signal["median_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["1-3", "4-10", "11-20", "21+"],
    include_lowest=True,
)

# March-only percentile rank of CTR among pages with similar search position.
# method='average' handles ties (including zero CTR) without inventing ordering.
ctr_signal["ctr_within_position_percentile"] = (
    ctr_signal
    .groupby("position_band", observed=False)["aggregate_ctr"]
    .rank(method="average", pct=True)
)

ctr_signal["ctr_vs_position_bucket"] = pd.cut(
    ctr_signal["ctr_within_position_percentile"],
    bins=[0.0, 0.25, 0.50, 0.75, 1.0],
    labels=[
        "bottom 25%",
        "25-50%",
        "50-75%",
        "top 25%",
    ],
    include_lowest=True,
)

position_band_diagnostic = (
    ctr_signal
    .groupby("position_band", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        median_position=("median_position", "median"),
        median_ctr=("aggregate_ctr", "median"),
        zero_ctr_rate=("aggregate_ctr", lambda s: float((s == 0).mean())),
    )
    .reset_index()
)
position_band_diagnostic["zero_ctr_rate_pct"] = (
    100 * position_band_diagnostic.pop("zero_ctr_rate")
)

ctr_position_table = (
    ctr_signal
    .groupby("ctr_vs_position_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        decline_rate=("future_decline", "mean"),
        median_future_impression_change=("future_impression_change", "median"),
        mean_future_impression_change=("future_impression_change", "mean"),
        median_within_position_percentile=(
            "ctr_within_position_percentile", "median"
        ),
    )
    .reset_index()
)

ctr_position_table["decline_rate_pct"] = (
    100 * ctr_position_table.pop("decline_rate")
)

print("\nPOSITION-BAND DIAGNOSTIC (MARCH ONLY)")
display(position_band_diagnostic)

print("\nCTR-VS-POSITION SIGNAL TABLE")
display(ctr_position_table)


Locked POC clients: 21
Locked POC pages: 2520
Expected: 21 clients and 2,520 pages
Signal rows: 2520

STALENESS SIGNAL TABLE


,age_bucket,n,median_future_impression_change,mean_future_impression_change,decline_rate_pct
0,0-90 days,1023,-0.085986,0.262161,55.718475
1,91-180 days,265,-0.305471,-0.128728,72.075472
2,181-365 days,740,-0.224764,0.018090,65.810811
3,366+ days,492,-0.497090,-0.385274,84.959350



POSITION-BAND DIAGNOSTIC (MARCH ONLY)


,position_band,n,median_position,median_ctr,zero_ctr_rate_pct
0,1-3,159,2.323664,0.003486,18.238994
1,4-10,1545,6.223301,0.001688,29.514563
2,11-20,405,13.500000,0.000719,42.469136
3,21+,411,32.119048,0.000000,63.746959



CTR-VS-POSITION SIGNAL TABLE


,ctr_vs_position_bucket,n,median_future_impression_change,mean_future_impression_change,median_within_position_percentile,decline_rate_pct
0,bottom 25%,667,-0.346976,-0.197757,0.147896,74.662669
1,25-50%,648,-0.301211,-0.133100,0.327276,71.141975
2,50-75%,582,-0.252960,0.088778,0.637864,65.463918
3,top 25%,623,-0.047109,0.360181,0.877023,52.327448


## 2. Build the ranked queue (writes the CSV)

The queue contains only pages that meet the single low-CTR-for-position rule. `baseline_score` is **2** for a bottom-quartile CTR page and **3** when that same page is also 366+ days old. Ties are ordered by lower within-position CTR percentile first, then by the pseudonymized IDs only for deterministic ordering.

The actionable CSV contains March-safe fields only. April outcomes are merged separately inside the notebook only for evaluation and are not written into the recommendation queue.

### Executed baseline result

The rule queued **667 of 2,520 pages (26.47%)**. The full balanced population's observed April decline rate is **66.11%**, compared with **74.66%** among queued pages. The ranked queue produced **Precision@10 = 70%**, **Precision@20 = 80%**, and **Precision@50 = 92%**. There are **552 score-2 pages** and **115 score-3 pages**. These are evaluation results for this fixed rule, not causal claims.

The notebook writes `work/outputs/baseline_action_score.csv` and `work/outputs/baseline_metrics.json` on every run.


In [2]:
# STEP 2 — encode ONE transparent baseline rule and write the ranked queue.

from pathlib import Path
import json

# Bring March-safe content age into the CTR-relative-to-position frame.
rule_frame = ctr_signal.merge(
    age_frame,
    on=["client_hash_id", "content_hash_id"],
    how="left",
)

rule_frame["low_ctr_for_position"] = (
    rule_frame["ctr_within_position_percentile"] <= 0.25
)
rule_frame["stale_366_plus"] = (
    rule_frame["content_age_days"] >= 366
)

# Transparent fixed-point rule:
# 2 points = confirmed low-CTR-for-position condition.
# +1 point = oldest staleness bucket, used only as a priority boost.
rule_frame["baseline_score"] = (
    2 * rule_frame["low_ctr_for_position"].astype(int)
    + rule_frame["stale_366_plus"].astype(int)
)

queue = (
    rule_frame[rule_frame["low_ctr_for_position"]]
    .copy()
)

# Exactly one reason code and one action label for every queued item.
queue["reason_code"] = "LOW_CTR_FOR_POSITION"
queue["action_label"] = "REVIEW_CTR_REFRESH"

queue = queue.sort_values(
    [
        "baseline_score",
        "ctr_within_position_percentile",
        "client_hash_id",
        "content_hash_id",
    ],
    ascending=[False, True, True, True],
).reset_index(drop=True)

queue.insert(0, "rank", np.arange(1, len(queue) + 1))

queue_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "baseline_score",
    "reason_code",
    "action_label",
    "content_age_days",
    "stale_366_plus",
    "position_band",
    "median_position",
    "aggregate_ctr",
    "ctr_within_position_percentile",
]

output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)
queue_path = output_dir / "baseline_action_score.csv"
queue[queue_columns].to_csv(queue_path, index=False)

# Evaluation uses the future outcome only after the March-only queue is frozen.
evaluation = queue[
    ["client_hash_id", "content_hash_id", "baseline_score"]
].merge(
    signal_frame[
        [
            "client_hash_id",
            "content_hash_id",
            "future_decline",
            "future_impression_change",
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="left",
)

base_rate = float(signal_frame["future_decline"].mean())

def precision_at_k(frame, k):
    k_eff = min(k, len(frame))
    return float(frame.head(k_eff)["future_decline"].mean())

metrics = {
    "population_pages": int(len(rule_frame)),
    "queued_pages": int(len(queue)),
    "queued_share_pct": float(100 * len(queue) / len(rule_frame)),
    "population_decline_base_rate": base_rate,
    "queue_decline_rate": float(evaluation["future_decline"].mean()),
    "precision_at_10": precision_at_k(evaluation, 10),
    "precision_at_20": precision_at_k(evaluation, 20),
    "precision_at_50": precision_at_k(evaluation, 50),
    "reason_codes": sorted(queue["reason_code"].unique().tolist()),
    "action_labels": sorted(queue["action_label"].unique().tolist()),
    "score_counts": {
        str(int(k)): int(v)
        for k, v in queue["baseline_score"].value_counts().sort_index().items()
    },
}

metrics_path = output_dir / "baseline_metrics.json"
with open(metrics_path, "w", encoding="utf-8") as fh:
    json.dump(metrics, fh, indent=2)

print("BASELINE RULE")
print(
    "Queue bottom-quartile March CTR within position band; "
    "score 2, plus 1 point if age >=366 days."
)
print("Reason code:", queue["reason_code"].unique().tolist())
print("Action label:", queue["action_label"].unique().tolist())
print("Population pages:", len(rule_frame))
print("Queued pages:", len(queue))
print(f"Queued share: {100 * len(queue) / len(rule_frame):.2f}%")
print(f"Population decline base rate: {100 * base_rate:.2f}%")
print(f"Queue decline rate: {100 * evaluation['future_decline'].mean():.2f}%")
print(f"Precision@10: {100 * metrics['precision_at_10']:.2f}%")
print(f"Precision@20: {100 * metrics['precision_at_20']:.2f}%")
print(f"Precision@50: {100 * metrics['precision_at_50']:.2f}%")
print("Score counts:", metrics["score_counts"])
print("CSV written:", queue_path)
print("Metrics written:", metrics_path)

display(queue[queue_columns].head(20))


BASELINE RULE
Queue bottom-quartile March CTR within position band; score 2, plus 1 point if age >=366 days.
Reason code: ['LOW_CTR_FOR_POSITION']
Action label: ['REVIEW_CTR_REFRESH']
Population pages: 2520
Queued pages: 667
Queued share: 26.47%
Population decline base rate: 66.11%
Queue decline rate: 74.66%
Precision@10: 70.00%
Precision@20: 80.00%
Precision@50: 92.00%
Score counts: {'2': 552, '3': 115}
CSV written: ../outputs/baseline_action_score.csv
Metrics written: ../outputs/baseline_metrics.json


,rank,client_hash_id,content_hash_id,baseline_score,reason_code,action_label,content_age_days,stale_366_plus,position_band,median_position,aggregate_ctr,ctr_within_position_percentile
0,1,client_e547b89c05043229,content_01a36626f8574f77,3,LOW_CTR_FOR_POSITION,REVIEW_CTR_REFRESH,417.0,True,1-3,2.777778,0.0,0.094340
1,2,client_65de48885f4ef01b,content_0110b9e0f571c404,3,LOW_CTR_FOR_POSITION,REVIEW_CTR_REFRESH,375.0,True,4-10,8.100758,0.0,0.147896
2,3,client_65de48885f4ef01b,content_13d6f8d3a22e2594,3,LOW_CTR_FOR_POSITION,REVIEW_CTR_REFRESH,375.0,True,4-10,9.375000,0.0,0.147896
3,4,client_65de48885f4ef01b,content_1490255ba6c6e0f5,3,LOW_CTR_FOR_POSITION,REVIEW_CTR_REFRESH,375.0,True,4-10,8.450000,0.0,0.147896
4,5,client_65de48885f4ef01b,content_1e56be877b9b2bfc,3,LOW_CTR_FOR_POSITION,REVIEW_CTR_REFRESH,375.0,True,4-10,7.615385,0.0,0.147896
5,6,client_65de48885f4ef01b,content_25323bb1cd2093e8,3,LOW_CTR_FOR_POSITION,REVIEW_CTR_REFRESH,375.0,True,4-10,7.846154,0.0,0.147896
6,7,client_65de48885f4ef01b,content_2942a29b5bafcdfb,3,LOW_CTR_FOR_POSITION,REVIEW_CTR_REFRESH,375.0,True,4-10,6.138889,0.0,0.147896
7,8,client_65de48885f4ef01b,content_2d4a9a63aeded404,3,LOW_CTR_FOR_POSITION,REVIEW_CTR_REFRESH,375.0,True,4-10,7.625000,0.0,0.147896
8,9,client_65de48885f4ef01b,content_359c3c32e917bf79,3,LOW_CTR_FOR_POSITION,REVIEW_CTR_REFRESH,375.0,True,4-10,6.439716,0.0,0.147896
9,10,client_65de48885f4ef01b,content_411cc85a2063de01,3,LOW_CTR_FOR_POSITION,REVIEW_CTR_REFRESH,375.0,True,4-10,5.793599,0.0,0.147896


## 3. Top-10 skeptical review

The assignment requires ten reviewed rows; the sibling/full audit can go to top-20, but that is optional. The review below does **not** change the ranking. It inspects the already-frozen March-only queue and asks whether each recommendation could be misleading.

For every top-10 row the notebook prints: the action, why it is there, a confidence note, and what would make it wrong.

### Executed review finding

All ten top-ranked pages have **0% March CTR** and a baseline score of **3**, so every one combines the confirmed low-CTR-for-position condition with the 366+ day staleness boost. Rank #1 is a **417-day-old** page in position band **1–3** at the **9.43th within-position CTR percentile**. Ranks #2–#10 are **375-day-old** pages in position band **4–10** at the **14.79th percentile**. This concentration is useful but also a warning: many top rows are tied on the same rule evidence, so hashed-ID ordering is only deterministic tie-breaking, not evidence that #2 is substantively better than #10.

The human-review caveat is therefore explicit: a zero CTR can still be misleading if exposure/query mix is sparse or unusual, or if the true constraint is ranking rather than title/snippet/content quality. The rule identifies review candidates; it does not prove a refresh will improve performance.


In [3]:
# STEP 3 — review the actual top 10 without changing the frozen ranking.

top10 = queue.head(10).copy()

# Bring in March exposure only as review context; it does not affect the score or rank.
review_context = balanced_poc[
    [
        "client_hash_id",
        "content_hash_id",
        "exposure_tier",
        "march_avg_impressions_per_day",
    ]
].copy()

top10 = top10.merge(
    review_context,
    on=["client_hash_id", "content_hash_id"],
    how="left",
)

def pct(x):
    return f"{100 * float(x):.2f}%"

def review_why(row):
    return (
        f"March CTR {pct(row['aggregate_ctr'])} is at the "
        f"{pct(row['ctr_within_position_percentile'])} within-position percentile "
        f"for position band {row['position_band']}; age is "
        f"{int(row['content_age_days'])} days, so the page receives score "
        f"{int(row['baseline_score'])}."
    )

def review_confidence(row):
    return (
        "Moderate: low CTR relative to position was CONFIRMED; "
        "the 366+ day age point is only a MIXED-signal priority boost."
    )

def review_wrong(row):
    if float(row["aggregate_ctr"]) == 0 and str(row["position_band"]) == "21+":
        return (
            "The recommendation would be weak if zero CTR is mainly expected from "
            "deep ranking rather than a title/snippet/content problem; ranking improvement "
            "could be the real action."
        )
    if float(row["aggregate_ctr"]) == 0:
        return (
            "The recommendation would be weak if zero CTR is driven by sparse/noisy "
            "March exposure or query mix rather than an actionable title/snippet/content issue."
        )
    if str(row["exposure_tier"]) == "Low":
        return (
            "The recommendation would be weak if the low relative CTR is unstable because "
            "this page has low March exposure, or if manual SERP/query-intent review shows "
            "the observed CTR is appropriate."
        )
    return (
        "The recommendation would be weak if manual SERP/query-intent review shows the "
        "CTR is appropriate for the page's actual query mix, or the title/snippet/content "
        "is already aligned and another ranking constraint is the real issue."
    )

top10_review = top10[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "action_label",
        "reason_code",
        "baseline_score",
        "content_age_days",
        "position_band",
        "aggregate_ctr",
        "ctr_within_position_percentile",
        "exposure_tier",
        "march_avg_impressions_per_day",
    ]
].copy()

top10_review["action"] = top10_review["action_label"]
top10_review["why_it_is_here"] = top10.apply(review_why, axis=1)
top10_review["confidence_note"] = top10.apply(review_confidence, axis=1)
top10_review["what_would_make_it_wrong"] = top10.apply(review_wrong, axis=1)

review_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "action",
    "reason_code",
    "why_it_is_here",
    "confidence_note",
    "what_would_make_it_wrong",
]

assert len(top10_review) == 10
assert top10_review["rank"].tolist() == list(range(1, 11))
assert top10_review["action"].nunique() == 1
assert top10_review["reason_code"].nunique() == 1

print("TOP-10 REVIEW ROWS:", len(top10_review))
print("Ranking preserved:", top10_review["rank"].tolist() == list(range(1, 11)))
print("Unique actions:", top10_review["action"].unique().tolist())
print("Unique reason codes:", top10_review["reason_code"].unique().tolist())
print("\nTOP-10 SKEPTICAL REVIEW")
display(top10_review[review_columns])

print("\nONE-LINE REVIEW PER PICK")
for _, row in top10_review[review_columns].iterrows():
    print(
        f"#{int(row['rank'])} | {row['action']} | "
        f"{row['why_it_is_here']} | "
        f"Confidence: {row['confidence_note']} | "
        f"Wrong if: {row['what_would_make_it_wrong']}"
    )


TOP-10 REVIEW ROWS: 10
Ranking preserved: True
Unique actions: ['REVIEW_CTR_REFRESH']
Unique reason codes: ['LOW_CTR_FOR_POSITION']

TOP-10 SKEPTICAL REVIEW


,rank,client_hash_id,content_hash_id,action,reason_code,why_it_is_here,confidence_note,what_would_make_it_wrong
0,1,client_e547b89c05043229,content_01a36626f8574f77,REVIEW_CTR_REFRESH,LOW_CTR_FOR_POSITION,March CTR 0.00% is at the 9.43% within-positio...,Moderate: low CTR relative to position was CON...,The recommendation would be weak if zero CTR i...
1,2,client_65de48885f4ef01b,content_0110b9e0f571c404,REVIEW_CTR_REFRESH,LOW_CTR_FOR_POSITION,March CTR 0.00% is at the 14.79% within-positi...,Moderate: low CTR relative to position was CON...,The recommendation would be weak if zero CTR i...
2,3,client_65de48885f4ef01b,content_13d6f8d3a22e2594,REVIEW_CTR_REFRESH,LOW_CTR_FOR_POSITION,March CTR 0.00% is at the 14.79% within-positi...,Moderate: low CTR relative to position was CON...,The recommendation would be weak if zero CTR i...
3,4,client_65de48885f4ef01b,content_1490255ba6c6e0f5,REVIEW_CTR_REFRESH,LOW_CTR_FOR_POSITION,March CTR 0.00% is at the 14.79% within-positi...,Moderate: low CTR relative to position was CON...,The recommendation would be weak if zero CTR i...
4,5,client_65de48885f4ef01b,content_1e56be877b9b2bfc,REVIEW_CTR_REFRESH,LOW_CTR_FOR_POSITION,March CTR 0.00% is at the 14.79% within-positi...,Moderate: low CTR relative to position was CON...,The recommendation would be weak if zero CTR i...
5,6,client_65de48885f4ef01b,content_25323bb1cd2093e8,REVIEW_CTR_REFRESH,LOW_CTR_FOR_POSITION,March CTR 0.00% is at the 14.79% within-positi...,Moderate: low CTR relative to position was CON...,The recommendation would be weak if zero CTR i...
6,7,client_65de48885f4ef01b,content_2942a29b5bafcdfb,REVIEW_CTR_REFRESH,LOW_CTR_FOR_POSITION,March CTR 0.00% is at the 14.79% within-positi...,Moderate: low CTR relative to position was CON...,The recommendation would be weak if zero CTR i...
7,8,client_65de48885f4ef01b,content_2d4a9a63aeded404,REVIEW_CTR_REFRESH,LOW_CTR_FOR_POSITION,March CTR 0.00% is at the 14.79% within-positi...,Moderate: low CTR relative to position was CON...,The recommendation would be weak if zero CTR i...
8,9,client_65de48885f4ef01b,content_359c3c32e917bf79,REVIEW_CTR_REFRESH,LOW_CTR_FOR_POSITION,March CTR 0.00% is at the 14.79% within-positi...,Moderate: low CTR relative to position was CON...,The recommendation would be weak if zero CTR i...
9,10,client_65de48885f4ef01b,content_411cc85a2063de01,REVIEW_CTR_REFRESH,LOW_CTR_FOR_POSITION,March CTR 0.00% is at the 14.79% within-positi...,Moderate: low CTR relative to position was CON...,The recommendation would be weak if zero CTR i...



ONE-LINE REVIEW PER PICK
#1 | REVIEW_CTR_REFRESH | March CTR 0.00% is at the 9.43% within-position percentile for position band 1-3; age is 417 days, so the page receives score 3. | Confidence: Moderate: low CTR relative to position was CONFIRMED; the 366+ day age point is only a MIXED-signal priority boost. | Wrong if: The recommendation would be weak if zero CTR is driven by sparse/noisy March exposure or query mix rather than an actionable title/snippet/content issue.
#2 | REVIEW_CTR_REFRESH | March CTR 0.00% is at the 14.79% within-position percentile for position band 4-10; age is 375 days, so the page receives score 3. | Confidence: Moderate: low CTR relative to position was CONFIRMED; the 366+ day age point is only a MIXED-signal priority boost. | Wrong if: The recommendation would be weak if zero CTR is driven by sparse/noisy March exposure or query mix rather than an actionable title/snippet/content issue.
#3 | REVIEW_CTR_REFRESH | March CTR 0.00% is at the 14.79% within-posi

## 4. Weak picks + leakage check

This section does two things: (1) identify queue entries that deserve extra human skepticism, without changing the frozen ranking, and (2) verify that the actionable baseline uses only March-safe inputs.

A **weak/fragile pick** here does not mean the page is definitely wrong. It means the recommendation is easier to misread because the observed March evidence is thin or potentially explained by a non-content factor such as low exposure or deep ranking.

### Executed weak-pick finding

Of the **667 queued pages**, **657 (98.50%)** have zero March CTR. **389 (58.32%)** are also in the Low exposure tier, so those same 389 pages carry at least two review-fragility flags: zero CTR plus low exposure. No queued page is in the **21+** position band, so deep ranking is not the observed fragility in this queue. The highest-ranked fragile examples therefore deserve manual checking of impression volume, query mix, and SERP intent before a refresh is acted on. These flags are review context only and do not alter the baseline score or ranking.

### Leakage audit — PASS

The executed programmatic check found **no forbidden target/future fields** in either the scoring/ranking inputs or `baseline_action_score.csv`. The score uses March aggregate CTR, March median position, March-derived within-position CTR percentile, and content age measured as of **2026-03-31**; pseudonymized IDs are used only for deterministic tie-breaking. April/future values are joined only after the queue is frozen for evaluation and never enter the score, rank, reason code, action label, or actionable CSV.


In [4]:
# STEP 4 — identify fragile picks and prove the queue is leakage-safe.

# 4A. Weak/fragile-pick audit.
weak_pick_frame = queue.merge(
    balanced_poc[
        [
            "client_hash_id",
            "content_hash_id",
            "exposure_tier",
            "march_avg_impressions_per_day",
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="left",
)

# Transparent fragility flags for review only; none alter the score or rank.
weak_pick_frame["zero_ctr"] = weak_pick_frame["aggregate_ctr"] == 0
weak_pick_frame["deep_position"] = (
    weak_pick_frame["position_band"].astype(str) == "21+"
)
weak_pick_frame["low_exposure_context"] = (
    weak_pick_frame["exposure_tier"].astype(str) == "Low"
)

weak_pick_frame["fragility_flag_count"] = (
    weak_pick_frame[
        ["zero_ctr", "deep_position", "low_exposure_context"]
    ]
    .astype(int)
    .sum(axis=1)
)

fragility_summary = pd.DataFrame({
    "flag": [
        "zero_ctr",
        "deep_position",
        "low_exposure_context",
        "two_or_more_fragility_flags",
    ],
    "n": [
        int(weak_pick_frame["zero_ctr"].sum()),
        int(weak_pick_frame["deep_position"].sum()),
        int(weak_pick_frame["low_exposure_context"].sum()),
        int((weak_pick_frame["fragility_flag_count"] >= 2).sum()),
    ],
})
fragility_summary["pct_of_queue"] = (
    100 * fragility_summary["n"] / len(weak_pick_frame)
)

# Show the highest-ranked fragile examples without changing queue order.
weak_examples = weak_pick_frame[
    weak_pick_frame["fragility_flag_count"] >= 2
][
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "baseline_score",
        "position_band",
        "aggregate_ctr",
        "ctr_within_position_percentile",
        "content_age_days",
        "exposure_tier",
        "march_avg_impressions_per_day",
        "zero_ctr",
        "deep_position",
        "low_exposure_context",
        "fragility_flag_count",
    ]
].head(10)

print("FRAGILITY SUMMARY")
display(fragility_summary)

print("\nHIGHEST-RANKED FRAGILE EXAMPLES")
display(weak_examples)

# 4B. Leakage audit.
# Explicitly enumerate the fields that determine eligibility, score, ranking,
# reason code, and action label.
baseline_input_fields = {
    "aggregate_ctr": "March-only",
    "median_position": "March-only",
    "position_band": "derived from March median_position",
    "ctr_within_position_percentile": "derived only from March CTR within March position band",
    "content_age_days": "static metadata measured as of 2026-03-31",
    "stale_366_plus": "derived from March-as-of content age",
    "client_hash_id": "tie-break only; not predictive",
    "content_hash_id": "tie-break only; not predictive",
}

def is_forbidden_field_name(field):
    """Detect future/target-derived field names without flagging action_label."""
    name = field.lower()
    if name == "action_label":
        return False
    return (
        name.startswith("april_")
        or name.startswith("future_")
        or name.endswith("_label")
        or name in {"trend_pct", "trend_direction"}
        or name.startswith("is_declining")
    )

scoring_fields = [
    "aggregate_ctr",
    "median_position",
    "position_band",
    "ctr_within_position_percentile",
    "content_age_days",
    "stale_366_plus",
    "client_hash_id",
    "content_hash_id",
]

leaky_scoring_fields = [
    field
    for field in scoring_fields
    if is_forbidden_field_name(field)
]

queue_file_columns = list(queue[queue_columns].columns)
leaky_queue_columns = [
    field
    for field in queue_file_columns
    if is_forbidden_field_name(field)
]

# Ensure outcome/evaluation columns are absent from the actionable queue.
forbidden_exact = {
    "future_decline",
    "future_impression_change",
    "april_avg_impressions_per_day",
    "is_declining_label",
    "trend_pct",
    "trend_direction",
}

forbidden_in_queue = sorted(forbidden_exact.intersection(queue_file_columns))

# Programmatic assertions: fail the notebook if leakage appears.
assert leaky_scoring_fields == []
assert leaky_queue_columns == []
assert forbidden_in_queue == []

print("\nLEAKAGE CHECK")
print("Baseline scoring/ranking inputs:")
for field, provenance in baseline_input_fields.items():
    print(f"- {field}: {provenance}")

print("\nForbidden target/future scoring fields:", leaky_scoring_fields)
print("Forbidden target/future queue columns:", leaky_queue_columns)
print("Forbidden exact fields in queue:", forbidden_in_queue)
print("LEAKAGE VERDICT: PASS")
print(
    "April/future values are used only in evaluation objects after the "
    "March-only queue is frozen; they do not enter the score, rank, reason code, "
    "action label, or baseline_action_score.csv."
)


FRAGILITY SUMMARY


,flag,n,pct_of_queue
0,zero_ctr,657,98.50075
1,deep_position,0,0.00000
2,low_exposure_context,389,58.32084
3,two_or_more_fragility_flags,389,58.32084



HIGHEST-RANKED FRAGILE EXAMPLES


,rank,client_hash_id,content_hash_id,baseline_score,position_band,aggregate_ctr,ctr_within_position_percentile,content_age_days,exposure_tier,march_avg_impressions_per_day,zero_ctr,deep_position,low_exposure_context,fragility_flag_count
0,1,client_e547b89c05043229,content_01a36626f8574f77,3,1-3,0.0,0.094340,417.0,Low,9.137931,True,False,True,2
2,3,client_65de48885f4ef01b,content_13d6f8d3a22e2594,3,4-10,0.0,0.147896,375.0,Low,11.806452,True,False,True,2
3,4,client_65de48885f4ef01b,content_1490255ba6c6e0f5,3,4-10,0.0,0.147896,375.0,Low,5.000000,True,False,True,2
4,5,client_65de48885f4ef01b,content_1e56be877b9b2bfc,3,4-10,0.0,0.147896,375.0,Low,13.225806,True,False,True,2
10,11,client_73cda7b4e4f265ea,content_00613ea10efe8893,3,4-10,0.0,0.147896,393.0,Low,7.903226,True,False,True,2
11,12,client_73cda7b4e4f265ea,content_0065965b3bdb2904,3,4-10,0.0,0.147896,412.0,Low,3.142857,True,False,True,2
15,16,client_9958f0a7ae1df715,content_00a18828280a576d,3,4-10,0.0,0.147896,368.0,Low,3.875000,True,False,True,2
16,17,client_9958f0a7ae1df715,content_00e33c6186c5df0c,3,4-10,0.0,0.147896,447.0,Low,10.645161,True,False,True,2
23,24,client_c182d11e4862a37d,content_004b73353d26c47c,3,4-10,0.0,0.147896,487.0,Low,2.608696,True,False,True,2
26,27,client_c182d11e4862a37d,content_02db8adb0b582966,3,4-10,0.0,0.147896,487.0,Low,12.555556,True,False,True,2



LEAKAGE CHECK
Baseline scoring/ranking inputs:
- aggregate_ctr: March-only
- median_position: March-only
- position_band: derived from March median_position
- ctr_within_position_percentile: derived only from March CTR within March position band
- content_age_days: static metadata measured as of 2026-03-31
- stale_366_plus: derived from March-as-of content age
- client_hash_id: tie-break only; not predictive
- content_hash_id: tie-break only; not predictive

Forbidden target/future scoring fields: []
Forbidden target/future queue columns: []
Forbidden exact fields in queue: []
LEAKAGE VERDICT: PASS
April/future values are used only in evaluation objects after the March-only queue is frozen; they do not enter the score, rank, reason code, action label, or baseline_action_score.csv.


## 5. Shared client-grouped holdout for all three baselines

Assignment 4 already established the honest validation design used in its leakage demonstration: `GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)` grouped by `client_hash_id`. Assignment 5 freezes that same split for **classification, regression, and ranking** so Assignment 6 inherits one unchanged benchmark population.

### Executed split

- Population: **2,520 pages across 21 clients**.
- Train: **1,800 pages across 15 clients**.
- Test: **720 pages across 6 clients**.
- Client overlap: **0**.

The split also exposes a real cross-client distribution shift that must not be hidden: observed decline prevalence is **75.11% in train versus 43.61% in test**. This makes the held-out benchmark harder, but it is exactly why the client-grouped design is useful: Assignment 6 must demonstrate generalisation to unseen clients rather than benefit from page-level leakage across the same clients.

Targets are locked as:

- **Classification:** `future_decline = 1` when `future_impression_change < 0`, otherwise `0`.
- **Regression:** continuous `future_impression_change`.
- **Ranking relevance:** the same binary `future_decline` outcome at the already-locked `K = 50`.

The exact pseudonymized train/test client lists are persisted in `work/outputs/baseline_split_manifest.json`.


In [5]:
# STEP 5 — freeze the shared client-grouped train/test split.

from sklearn.model_selection import GroupShuffleSplit

benchmark_frame = signal_frame[
    [
        "client_hash_id",
        "content_hash_id",
        "future_impression_change",
        "future_decline",
    ]
].copy()

groups = benchmark_frame["client_hash_id"]
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(
        benchmark_frame,
        benchmark_frame["future_impression_change"],
        groups=groups,
    )
)

train_frame = benchmark_frame.iloc[train_idx].copy()
test_frame = benchmark_frame.iloc[test_idx].copy()

train_clients = sorted(train_frame["client_hash_id"].unique().tolist())
test_clients = sorted(test_frame["client_hash_id"].unique().tolist())

assert set(train_clients).isdisjoint(test_clients)
assert len(train_frame) + len(test_frame) == len(benchmark_frame)
assert set(train_frame["client_hash_id"]) == set(train_clients)
assert set(test_frame["client_hash_id"]) == set(test_clients)

split_manifest = {
    "splitter": "GroupShuffleSplit",
    "group_field": "client_hash_id",
    "test_size": 0.25,
    "random_state": 42,
    "population_pages": int(len(benchmark_frame)),
    "population_clients": int(benchmark_frame["client_hash_id"].nunique()),
    "train_pages": int(len(train_frame)),
    "test_pages": int(len(test_frame)),
    "train_clients": train_clients,
    "test_clients": test_clients,
    "client_overlap": [],
    "classification_target": "future_impression_change < 0",
    "regression_target": "future_impression_change",
    "ranking_relevance": "future_impression_change < 0",
    "ranking_k": 50,
}

split_manifest_path = output_dir / "baseline_split_manifest.json"
with open(split_manifest_path, "w", encoding="utf-8") as fh:
    json.dump(split_manifest, fh, indent=2)

print("SHARED CLIENT-GROUPED SPLIT")
print("Population pages:", len(benchmark_frame))
print("Population clients:", benchmark_frame["client_hash_id"].nunique())
print("Train pages:", len(train_frame))
print("Test pages:", len(test_frame))
print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(set(train_clients).intersection(test_clients)))
print(
    "Train decline prevalence:",
    f"{100 * train_frame['future_decline'].mean():.2f}%"
)
print(
    "Test decline prevalence:",
    f"{100 * test_frame['future_decline'].mean():.2f}%"
)
print("Split manifest written:", split_manifest_path)


SHARED CLIENT-GROUPED SPLIT
Population pages: 2520
Population clients: 21
Train pages: 1800
Test pages: 720
Train clients: 15
Test clients: 6
Client overlap: 0
Train decline prevalence: 75.11%
Test decline prevalence: 43.61%
Split manifest written: ../outputs/baseline_split_manifest.json


## 6. Frozen classification, regression, and held-out ranking baselines

These baselines intentionally contain **no learned feature relationships**. They are the minimum honest benchmarks that Assignment 6 must beat on the same held-out clients.

### Classification baseline — training prior / majority probability

Training decline prevalence is **75.11%**, so every held-out page receives probability **0.7511** and the majority-class prediction is decline. On the 720-page held-out set, where decline prevalence is **43.61%**, the frozen baseline achieves:

- **ROC-AUC = 0.500**
- Precision = **0.436**
- Recall = **1.000**
- F1 = **0.607**

ROC-AUC of 0.5 is expected for a constant-probability baseline; that is the point of this benchmark.

### Regression baseline — training-set mean future change

The training-set mean future impression change is **−0.1567**, while the held-out client mean is **+0.4722**, again showing the client-distribution shift. Predicting the training mean for every held-out page gives:

- **RMSE = 1.4311**
- MAE = **0.8125**
- Median Absolute Error = **0.3884**
- R² = **−0.2394**

The negative R² is valid: under this cross-client shift, the training-mean predictor is worse than simply predicting the held-out mean. Assignment 6 must still beat the frozen training-only baseline, not a test-informed value.

### Ranking baseline — frozen rule on the same held-out clients

The existing low-CTR-for-position rule with the 366+ day staleness boost is re-applied to the 720 held-out pages without changing the rule. It queues **166 pages**; 314 held-out pages are future declines. At `K = 50` the frozen ranking baseline is:

- **Precision@50 = 0.480**
- Recall@50 = **0.0764**
- Lift@50 = **1.1006**
- NDCG@50 = **0.4854**

The executed notebook displays one **Baseline Benchmark Table** containing all three tasks and writes the machine-readable receipt to `work/outputs/multitask_baseline_benchmark.json`.


In [6]:
# STEP 6 — compute all three frozen baselines on the shared held-out clients.

from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    mean_squared_error,
    mean_absolute_error,
    median_absolute_error,
    r2_score,
    ndcg_score,
)

# -------------------------
# Classification baseline
# -------------------------
y_cls_train = train_frame["future_decline"].astype(int).to_numpy()
y_cls_test = test_frame["future_decline"].astype(int).to_numpy()

train_decline_prior = float(y_cls_train.mean())
cls_prob = np.full(len(y_cls_test), train_decline_prior, dtype=float)
cls_pred = (cls_prob >= 0.5).astype(int)

classification_baseline = {
    "name": "training_prior_probability",
    "train_decline_prior": train_decline_prior,
    "majority_class": int(train_decline_prior >= 0.5),
    "test_decline_prevalence": float(y_cls_test.mean()),
    "roc_auc": float(roc_auc_score(y_cls_test, cls_prob)),
    "precision": float(precision_score(y_cls_test, cls_pred, zero_division=0)),
    "recall": float(recall_score(y_cls_test, cls_pred, zero_division=0)),
    "f1": float(f1_score(y_cls_test, cls_pred, zero_division=0)),
}

# -------------------------
# Regression baseline
# -------------------------
y_reg_train = train_frame["future_impression_change"].to_numpy(dtype=float)
y_reg_test = test_frame["future_impression_change"].to_numpy(dtype=float)

train_mean_future_change = float(y_reg_train.mean())
reg_pred = np.full(len(y_reg_test), train_mean_future_change, dtype=float)

regression_baseline = {
    "name": "training_mean_future_change",
    "train_mean_future_change": train_mean_future_change,
    "test_mean_future_change": float(y_reg_test.mean()),
    "rmse": float(np.sqrt(mean_squared_error(y_reg_test, reg_pred))),
    "mae": float(mean_absolute_error(y_reg_test, reg_pred)),
    "median_absolute_error": float(median_absolute_error(y_reg_test, reg_pred)),
    "r2": float(r2_score(y_reg_test, reg_pred)),
}

# -------------------------
# Ranking baseline
# -------------------------
# Re-apply the same frozen rule to the held-out client population.
heldout_rule = rule_frame[
    rule_frame["client_hash_id"].isin(test_clients)
][
    [
        "client_hash_id",
        "content_hash_id",
        "aggregate_ctr",
        "median_position",
        "content_age_days",
    ]
].copy()

heldout_rule["position_band"] = pd.cut(
    heldout_rule["median_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["1-3", "4-10", "11-20", "21+"],
    include_lowest=True,
)

heldout_rule["ctr_within_position_percentile"] = (
    heldout_rule
    .groupby("position_band", observed=False)["aggregate_ctr"]
    .rank(method="average", pct=True)
)

heldout_rule["low_ctr_for_position"] = (
    heldout_rule["ctr_within_position_percentile"] <= 0.25
)
heldout_rule["stale_366_plus"] = heldout_rule["content_age_days"] >= 366

# Preserve queue semantics: only low-CTR pages receive actionable score.
heldout_rule["baseline_score"] = np.where(
    heldout_rule["low_ctr_for_position"],
    2 + heldout_rule["stale_366_plus"].astype(int),
    0,
)

heldout_rank = heldout_rule.merge(
    test_frame[
        ["client_hash_id", "content_hash_id", "future_decline"]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner",
)

heldout_rank = heldout_rank.sort_values(
    [
        "baseline_score",
        "ctr_within_position_percentile",
        "client_hash_id",
        "content_hash_id",
    ],
    ascending=[False, True, True, True],
).reset_index(drop=True)

K = 50
topk = heldout_rank.head(K)
test_relevant = int(heldout_rank["future_decline"].sum())
topk_relevant = int(topk["future_decline"].sum())
test_base_rate = float(heldout_rank["future_decline"].mean())

precision_at_50 = float(topk["future_decline"].mean())
recall_at_50 = float(topk_relevant / test_relevant) if test_relevant else 0.0
lift_at_50 = float(precision_at_50 / test_base_rate) if test_base_rate else np.nan

# Numeric score mirrors the frozen lexicographic logic:
# score-3 > score-2 > non-queued; weaker within-position CTR ranks higher.
heldout_rank["ndcg_score_input"] = (
    heldout_rank["baseline_score"].astype(float)
    + np.where(
        heldout_rank["low_ctr_for_position"],
        (1.0 - heldout_rank["ctr_within_position_percentile"]) * 0.1,
        0.0,
    )
)

ndcg_at_50 = float(
    ndcg_score(
        heldout_rank["future_decline"].astype(int).to_numpy().reshape(1, -1),
        heldout_rank["ndcg_score_input"].to_numpy().reshape(1, -1),
        k=K,
        ignore_ties=False,
    )
)

ranking_baseline = {
    "name": "low_ctr_for_position_rule_with_staleness_boost",
    "test_pages": int(len(heldout_rank)),
    "test_relevant_pages": test_relevant,
    "queued_pages": int(heldout_rank["low_ctr_for_position"].sum()),
    "precision_at_50": precision_at_50,
    "recall_at_50": recall_at_50,
    "lift_at_50": lift_at_50,
    "ndcg_at_50": ndcg_at_50,
}

multi_task_benchmark = {
    "split": {
        "group_field": "client_hash_id",
        "test_size": 0.25,
        "random_state": 42,
        "train_pages": int(len(train_frame)),
        "test_pages": int(len(test_frame)),
        "train_clients": int(len(train_clients)),
        "test_clients": int(len(test_clients)),
    },
    "classification": classification_baseline,
    "regression": regression_baseline,
    "ranking": ranking_baseline,
}

benchmark_path = output_dir / "multitask_baseline_benchmark.json"
with open(benchmark_path, "w", encoding="utf-8") as fh:
    json.dump(multi_task_benchmark, fh, indent=2)

baseline_benchmark_table = pd.DataFrame([
    {
        "task": "Classification",
        "baseline": "Training prior probability / majority class",
        "primary_metric": "ROC-AUC",
        "primary_result": classification_baseline["roc_auc"],
        "supporting_metrics": (
            f"Precision={classification_baseline['precision']:.3f}; "
            f"Recall={classification_baseline['recall']:.3f}; "
            f"F1={classification_baseline['f1']:.3f}"
        ),
    },
    {
        "task": "Regression",
        "baseline": "Training mean future change",
        "primary_metric": "RMSE",
        "primary_result": regression_baseline["rmse"],
        "supporting_metrics": (
            f"MAE={regression_baseline['mae']:.3f}; "
            f"MedAE={regression_baseline['median_absolute_error']:.3f}; "
            f"R2={regression_baseline['r2']:.3f}"
        ),
    },
    {
        "task": "Ranking",
        "baseline": "Low CTR-for-position rule + 366d staleness boost",
        "primary_metric": "Precision@50",
        "primary_result": ranking_baseline["precision_at_50"],
        "supporting_metrics": (
            f"Recall@50={ranking_baseline['recall_at_50']:.3f}; "
            f"Lift@50={ranking_baseline['lift_at_50']:.3f}; "
            f"NDCG@50={ranking_baseline['ndcg_at_50']:.3f}"
        ),
    },
])

print("CLASSIFICATION BASELINE")
for k, v in classification_baseline.items():
    print(f"{k}: {v}")

print("\nREGRESSION BASELINE")
for k, v in regression_baseline.items():
    print(f"{k}: {v}")

print("\nRANKING BASELINE — SHARED HELD-OUT CLIENTS")
for k, v in ranking_baseline.items():
    print(f"{k}: {v}")

print("\nBASELINE BENCHMARK TABLE")
display(baseline_benchmark_table)

print("\nBenchmark receipt written:", benchmark_path)


CLASSIFICATION BASELINE
name: training_prior_probability
train_decline_prior: 0.7511111111111111
majority_class: 1
test_decline_prevalence: 0.4361111111111111
roc_auc: 0.5
precision: 0.4361111111111111
recall: 1.0
f1: 0.6073500967117988

REGRESSION BASELINE
name: training_mean_future_change
train_mean_future_change: -0.1567124448202151
test_mean_future_change: 0.4722108164076777
rmse: 1.4311128557344202
mae: 0.8124798805523925
median_absolute_error: 0.38840741218591746
r2: -0.23935552498275392

RANKING BASELINE — SHARED HELD-OUT CLIENTS
name: low_ctr_for_position_rule_with_staleness_boost
test_pages: 720
test_relevant_pages: 314
queued_pages: 166
precision_at_50: 0.48
recall_at_50: 0.07643312101910828
lift_at_50: 1.1006369426751592
ndcg_at_50: 0.4854295799894303

BASELINE BENCHMARK TABLE


,task,baseline,primary_metric,primary_result,supporting_metrics
0,Classification,Training prior probability / majority class,ROC-AUC,0.500000,Precision=0.436; Recall=1.000; F1=0.607
1,Regression,Training mean future change,RMSE,1.431113,MAE=0.812; MedAE=0.388; R2=-0.239
2,Ranking,Low CTR-for-position rule + 366d staleness boost,Precision@50,0.480000,Recall@50=0.076; Lift@50=1.101; NDCG@50=0.485



Benchmark receipt written: ../outputs/multitask_baseline_benchmark.json


## 7. Baseline freeze for Assignment 6

The three baselines and the client-grouped split are now **frozen**.

Assignment 6 must inherit:

- the same 15 training clients and 6 held-out clients;
- classification target `future_impression_change < 0` with primary metric **ROC-AUC**;
- regression target `future_impression_change` with primary metric **RMSE**;
- ranking relevance `future_impression_change < 0` with **K = 50** and primary metric **Precision@50**;
- the exact baseline values reported above.

No Assignment 6 model result may be used to retrospectively change these baselines or the split.


## Self-check

Final multi-task baseline audit completed against the committed GitHub Actions execution.

- [x] Original two-signal audit and ranking queue remain intact
- [x] Shared client-grouped split is frozen with zero client overlap
- [x] Classification prior/majority baseline is evaluated on held-out clients
- [x] Regression training-mean baseline is evaluated on held-out clients
- [x] Ranking rule baseline is re-evaluated on the same held-out clients at K=50
- [x] One benchmark table contains all three tasks
- [x] JSON split + benchmark receipts are committed
- [x] Notebook runs top to bottom with no errors

**Submission state:** Assignment 5 complete under the project methodology; submission pending.
